# NB_Setup_Meta
Run **once** before the pipeline. Creates the `meta`, `bronze`, `silver`, `gold` schemas, the config table that drives the pipeline,
the watermark table and the pipeline log.

Set `reset_all = True` only for a clean restart: it **drops every table/view in bronze, silver, gold, meta and deletes `Files/raw` and `Files/metadata`**.

In [1]:
reset_all = False   # True = wipe bronze/silver/gold/meta tables and Files/raw (one-time clean restart)

StatementMeta(, b8eb4f16-d9f0-4a81-903b-1dc0ab9dcabe, 3, Finished, Available, Finished, False)

In [2]:
for s in ["meta", "bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")

if reset_all:
    for schema in ["bronze", "silver", "gold", "meta"]:
        for row in spark.sql(f"SHOW TABLES IN {schema}").collect():
            name = f"{schema}.{row['tableName']}"
            try:
                spark.sql(f"DROP TABLE IF EXISTS {name}")
            except Exception:
                spark.sql(f"DROP VIEW IF EXISTS {name}")
            print("dropped", name)
    for folder in ["Files/raw", "Files/metadata"]:
        try:
            notebookutils.fs.rm(folder, True)
            print("deleted", folder)
        except Exception as e:
            print("skipped", folder, "-", e)
    print("RESET DONE")

StatementMeta(, b8eb4f16-d9f0-4a81-903b-1dc0ab9dcabe, 4, Finished, Available, Finished, False)

In [ ]:
# Create the metadata tables (idempotent, safe on a fresh Lakehouse and after reset_all)
spark.sql("""
CREATE TABLE IF NOT EXISTS meta.config_resources (
    order_id      INT,
    resource_name STRING,
    endpoint      STRING,
    page_size     INT,
    key_column    STRING,
    is_active     BOOLEAN
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS meta.resource_watermark (
    resource_name    STRING,
    last_loaded_date STRING,
    updated_at       TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS meta.pipeline_log (
    run_id          STRING,
    resource_name   STRING,
    load_date       STRING,
    layer           STRING,
    step            STRING,
    page_no         INT,
    api_url         STRING,
    api_call_time   TIMESTAMP,
    http_status     INT,
    rows_fetched    INT,
    rows_written    BIGINT,
    api_total       BIGINT,
    raw_file_path   STRING,
    data_saved_time TIMESTAMP,
    status          STRING,
    error_message   STRING
) USING DELTA
""")

print("meta tables ready")

In [3]:
spark.sql("DELETE FROM meta.config_resources")

spark.sql("""
INSERT INTO meta.config_resources
(order_id, resource_name, endpoint, page_size, key_column, is_active)
VALUES
    (1, 'Patient',     'https://hapi.fhir.org/baseR4', 100, 'id', true),
    (2, 'Encounter',   'https://hapi.fhir.org/baseR4', 100, 'id', true),
    (3, 'Observation', 'https://hapi.fhir.org/baseR4', 100, 'id', true),
    (4, 'Condition',   'https://hapi.fhir.org/baseR4', 100, 'id', true)
""")

display(spark.table("meta.config_resources").orderBy("order_id"))

StatementMeta(, b8eb4f16-d9f0-4a81-903b-1dc0ab9dcabe, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35af48e3-7a48-4d0a-a336-db7c5f4ef279)

In [4]:
spark.sql("DESCRIBE TABLE meta.config_resources").show(truncate=False)

StatementMeta(, b8eb4f16-d9f0-4a81-903b-1dc0ab9dcabe, 6, Finished, Available, Finished, False)

+-------------+---------+-------+
|col_name     |data_type|comment|
+-------------+---------+-------+
|order_id     |int      |NULL   |
|resource_name|string   |NULL   |
|endpoint     |string   |NULL   |
|page_size    |int      |NULL   |
|key_column   |string   |NULL   |
|is_active    |boolean  |NULL   |
+-------------+---------+-------+



In [5]:
display(spark.sql("""
SELECT *
FROM meta.config_resources
ORDER BY order_id
"""))

StatementMeta(, b8eb4f16-d9f0-4a81-903b-1dc0ab9dcabe, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4fab9a24-268d-4563-9187-092f5b73afba)